In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 기본 경로 및 데이터 설정
datasets = ['Kd', 'Ki']
base_path = '/data/rlawlsgurjh/work/gitlab/CUE/cue/results'
dta_base_path = '/data/rlawlsgurjh/work/gitlab/CUE/dta_results'

label_map = {
    'regression': ["Ssel", 'PI', "WS(alpha=2)", "RS(k=2)"]
}
regression_candidates = ['rmse', 'r2', 'pcc']

# 베이스라인 모델 경로 및 포맷 설정
baseline_configs = {
    'DeepDTA':     {'path': os.path.join(dta_base_path, 'DeepDTA'), 'format': 'dta_csv'},
    'GraphDTA':    {'path': os.path.join(dta_base_path, 'GraphDTA'), 'format': 'dta_csv'},
    'TEFDTA':      {'path': os.path.join(dta_base_path, 'TEFDTA'), 'format': 'dta_csv'},
    'HMM-DTA':     {'path': os.path.join(dta_base_path, 'HMM-DTA'), 'format': 'dta_csv'},
    'ADMET-Print': {'path': '/data/rlawlsgurjh/work/gitlab/CUE/admet-print/results', 'format': 'new_csv'},
    'ADMET-AI':    {'path': '/data/rlawlsgurjh/work/gitlab/CUE/admet_ai/results/chemprop_rdkit', 'format': 'admet_csv'},
    'MMFDL':       {'path': '/data/rlawlsgurjh/work/gitlab/CUE/MMFDL/results/SGD/selectivity', 'format': 'mmfdl_summary'},
    'MMFRL':       {'path': '/data/rlawlsgurjh/work/gitlab/CUE/mmfrl/results/v1', 'format': 'new_csv'}
}

# 그래프에 표시될 모델 순서 (추가됨)
model_order = ['DeepDTA', 'GraphDTA', 'TEFDTA', 'HMM-DTA', 'ADMET-Print', 'ADMET-AI', 'MMFDL', 'MMFRL', 'CMFE(Ours)']

In [2]:
def load_baseline_metric(model_name, dataset, target, config):
    """모델별 포맷에 맞춰 metrics 정보를 가져오고 이름 정규화"""
    fmt = config['format']
    base_dir = config['path']
    
    if model_name == 'MMFDL':
        metrics_path = os.path.join(base_dir, dataset, target, 'v1', 'summary_metric.csv')
    else:
        # ADMET-AI는 metric.csv, 나머지는 metrics.csv
        metrics_path = os.path.join(base_dir, dataset, target, 'metric.csv' if fmt == 'admet_csv' else 'metrics.csv')
        
    if not os.path.exists(metrics_path): return None
    
    try:
        df = pd.read_csv(metrics_path)
        res = {}
        
        if fmt == 'dta_csv':
            row = df[df.iloc[:, 0].str.contains(target.replace('_label', ''), na=False, regex=False)]
            if row.empty: row = df.iloc[:1]
            for col in row.columns:
                norm_col = col.lower()
                if norm_col in regression_candidates: res[norm_col] = row[col].values[0]
                    
        elif fmt == 'admet_csv':
            row = df[df.iloc[:, 0] == 'mean']
            if not row.empty:
                for col in row.columns:
                    norm_col = col.lower()
                    if norm_col in regression_candidates: res[norm_col] = row[col].values[0]
                        
        elif fmt == 'mmfdl_summary':
            for _, r in df.iterrows():
                norm_col = str(r['metric']).lower()
                if norm_col in regression_candidates: res[norm_col] = r['mean']
                    
        elif fmt == 'new_csv': # ADMET-Print, MMFRL의 공통 구조 (rmse_mean 등)
            row = df.iloc[0]
            for col in row.index:
                if col.endswith('_mean'):
                    norm_col = col[:-5].lower() # '_mean' 글자 제거
                    if norm_col in regression_candidates:
                        res[norm_col] = row[col]
                        
        return res
    except: return None

# 데이터 통합 수집 실행
raw_rows = []
for d_name in datasets:
    for target in label_map['regression']: # regression만 수행
        version = 'v1'
        
        # 1. CMFE(Ours) 로드
        m_path = os.path.join(base_path, d_name, target, version, 'CMFE', 'metrics.csv')
        if os.path.exists(m_path):
            df_m = pd.read_csv(m_path)
            mlp_row = df_m[df_m['model_name'] == 'mlp']
            if not mlp_row.empty:
                entry = {'Dataset': d_name, 'Target': target, 'Model': 'CMFE(Ours)'}
                for col in mlp_row.columns:
                    norm_col = col.lower()
                    if norm_col in regression_candidates:
                        entry[norm_col] = mlp_row[col].values[0]
                raw_rows.append(entry)
        
        # 2. 베이스라인 로드
        for b_name, b_config in baseline_configs.items():
            b_metrics = load_baseline_metric(b_name, d_name, target, b_config)
            if b_metrics:
                entry = {'Dataset': d_name, 'Target': target, 'Model': b_name}
                entry.update(b_metrics)
                raw_rows.append(entry)

df_total = pd.DataFrame(raw_rows)
print(f"✅ 데이터 로드 완료: {len(df_total)}개의 항목이 수집되었습니다.")

✅ 데이터 로드 완료: 72개의 항목이 수집되었습니다.


In [3]:
def plot_comparison_styled(df_plot, dataset, target_name, metric_name, output_dir):
    """지표 라벨은 지우되, Y축 눈금 수치와 배경은 유지한 최종 버전"""
    df_target = df_plot.set_index('Model').reindex(model_order).dropna(subset=[metric_name]).reset_index()
    if df_target.empty: return

    models = df_target['Model'].tolist()
    values = df_target[metric_name].tolist()
    
    # davis_Plot.ipynb와 동일한 모델 수(9개, 베이스라인 8개)이므로 해치 패턴도 통일한다.
    baseline_hatches = ['///', '\\\\', '|||', '---', '+++', 'xxx', '..', '|']
    
    y_min, y_max = min(values), max(values)
    margin = (y_max - y_min) * 0.45 if (y_max - y_min) != 0 else 0.1
    
    fig, ax = plt.subplots(figsize=(6, 4)) # 모델이 많아졌으므로 너비를 살짝 늘림(6->7)
    x = np.arange(len(models))
    
    colors = ['lightgray'] * len(models)
    if 'CMFE(Ours)' in models:
        colors[models.index('CMFE(Ours)')] = 'salmon'
    
    bars = ax.bar(x, values, color=colors, edgecolor='black', alpha=0.9, linewidth=1)
    
    for i, bar in enumerate(bars):
        model_name = models[i]
        if model_name != 'CMFE(Ours)':
            bar.set_hatch(baseline_hatches[i % len(baseline_hatches)])
        
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + (margin * 0.03), 
                f'{height:.3f}', ha='center', va='bottom', fontsize=12,  # 폰트 사이즈 살짝 축소
                fontweight='bold', rotation=90)
    
    ax.set_title("")
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticklabels([])
    
    ax.tick_params(axis='y', labelsize=11)
    ax.tick_params(axis='x', which='both', length=0)
    
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    for spine in ax.spines.values():
        spine.set_visible(True)
    
    ax.set_ylim(max(0, y_min - (margin*0.1)), y_max + margin)
    
    plt.tight_layout()
    os.makedirs(output_dir, exist_ok=True)
    save_path = os.path.join(output_dir, f"{target_name}_{metric_name}.png")
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"✅ 저장 완료: {save_path}")

In [4]:
plot_base_dir = os.path.join(base_path, 'plots', 'benchmark_comparison')
for d_name in datasets:
    for target in label_map['regression']:
        df_target = df_total[(df_total['Dataset'] == d_name) & (df_total['Target'] == target)]
        if df_target.empty: continue

        g_name = 'DIS' if d_name == 'Kd' else 'INH'
        
        output_path = os.path.join(plot_base_dir, g_name)
        for m in regression_candidates:
            if m in df_target.columns:
                plot_comparison_styled(df_target, d_name, target, m, output_path)

✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/benchmark_comparison/DIS/Ssel_rmse.png


/tmp/ipykernel_2379813/3932917773.py:48: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/benchmark_comparison/DIS/Ssel_r2.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/benchmark_comparison/DIS/Ssel_pcc.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/benchmark_comparison/DIS/PI_rmse.png


/tmp/ipykernel_2379813/3932917773.py:48: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/benchmark_comparison/DIS/PI_r2.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/benchmark_comparison/DIS/PI_pcc.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/benchmark_comparison/DIS/WS(alpha=2)_rmse.png


/tmp/ipykernel_2379813/3932917773.py:48: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/benchmark_comparison/DIS/WS(alpha=2)_r2.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/benchmark_comparison/DIS/WS(alpha=2)_pcc.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/benchmark_comparison/DIS/RS(k=2)_rmse.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/benchmark_comparison/DIS/RS(k=2)_r2.png


/tmp/ipykernel_2379813/3932917773.py:48: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/benchmark_comparison/DIS/RS(k=2)_pcc.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/benchmark_comparison/INH/Ssel_rmse.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/benchmark_comparison/INH/Ssel_r2.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/benchmark_comparison/INH/Ssel_pcc.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/benchmark_comparison/INH/PI_rmse.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/benchmark_comparison/INH/PI_r2.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/benchmark_comparison/INH/PI_pcc.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/benchmark_comparison/INH/WS(alpha=2)_rmse.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/benchmark_comparison/INH/WS(alpha=2)_r2.png


/tmp/ipykernel_2379813/3932917773.py:48: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/benchmark_comparison/INH/WS(alpha=2)_pcc.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/benchmark_comparison/INH/RS(k=2)_rmse.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/benchmark_comparison/INH/RS(k=2)_r2.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/benchmark_comparison/INH/RS(k=2)_pcc.png
